# 🏋️ Day 4 실습 — Function Calling & 도구 연결

📖 강의 연계: Day 4 강의교안 전체 (모듈 4-1 ~ 4-4)

✅ **완료 기준**
- [ ] `@tool` 정의 → 4단계 루프를 직접 완성한다
- [ ] `tool_calls` JSON 구조를 셀 출력으로 단계별 관찰한다
- [ ] description 품질이 도구 선택 정확도에 영향을 줌을 실험으로 확인한다
- [ ] 내 마이 서비스 조각에 도구 1개 이상을 연결한 완전한 루프를 완성한다
- [ ] (⭐ 선택) Tavily 웹 검색 도구를 `@tool`로 래핑해 AI에게 실시간 정보를 제공한다

⚠️ 우측 상단 커널이 **`.venv`** 인지 확인하세요 (Colab 아님)

In [2]:
# 환경 점검 — 이 셀이 "환경 준비 완료"를 출력해야 다음 셀로 진행합니다
from dotenv import load_dotenv
import os

load_dotenv()

assert os.getenv("OPENAI_API_KEY"), (
    "❌ OPENAI_API_KEY 없음 — .env 파일 확인 (📖 강의교안 모듈 1-4 참조)"
)
if not os.getenv("LANGCHAIN_API_KEY"):
    print("⚠️  LANGCHAIN_API_KEY 없음 — LangSmith 트레이스가 기록되지 않습니다")

# Tavily는 선택 — 없어도 Step 1~4 실습 가능, Step 5에서 필요
TAVILY_AVAILABLE = bool(os.getenv("TAVILY_API_KEY"))

print("✅ 환경 준비 완료")
print(f"   API 키 앞 7자: {os.getenv('OPENAI_API_KEY')[:7]}...")
print(f"   Tavily 키   : {'✅ 있음' if TAVILY_AVAILABLE else '⚠️  없음 (Step 5에서 발급 안내)'}")

✅ 환경 준비 완료
   API 키 앞 7자: sk-proj...
   Tavily 키   : ✅ 있음


In [3]:
# LLM 공통 초기화 — 이후 모든 Step에서 재사용합니다
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage, SystemMessage
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
print("✅ LLM 초기화 완료:", llm.model_name)

✅ LLM 초기화 완료: gpt-4o-mini


## Step 1. @tool 정의 & 도구 스키마 확인

📖 강의 연계: 모듈 4-2 "v1: 단일 도구 정의" / 모듈 4-1 "Day 3 Pydantic과의 연결"

`@tool` 데코레이터는 함수의 **타입 힌트 + docstring**에서 Pydantic 기반 ArgsSchema를 자동 생성합니다.  
AI는 이 스키마를 읽고 언제, 어떤 인자로 도구를 호출할지 결정합니다.

아래 셀에서 스키마가 어떻게 생겼는지 직접 출력해 봅니다.

In [12]:
# ① 그대로 실행 — 강의의 코드를 그대로 실행해 성공 경험 확보
# 📖 강의 연계: 모듈 4-2 "@tool 기본 구현 — v1"

@tool
def get_employee_info(employee_id: str) -> dict:
    """
    직원 정보를 DB에서 조회합니다.

    사용 시점: 특정 직원의 이름, 부서, 직급 정보가 필요할 때.
    주의: 직원 ID는 반드시 'EMP' + 3자리 숫자 형식 (예: EMP001)

    Args:
        employee_id: 직원 고유 ID (형식: EMP + 3자리 숫자)
    Returns:
        직원 정보 dict: name, dept, level 키 포함
    """
    db = {
        "EMP001": {"name": "김철수", "dept": "AI개발팀",  "level": "팀장"},
        "EMP002": {"name": "이영희", "dept": "기획팀",    "level": "PM"},
        "EMP003": {"name": "박민준", "dept": "데이터팀",  "level": "시니어"},
    }
    return db.get(employee_id, {"error": f"직원 없음: {employee_id}"})

# AI에게 전달되는 스키마 확인 — Day 3 타입 힌트 → Pydantic 스키마 변환
import json as _json
schema = get_employee_info.args_schema.model_json_schema()
print("📋 AI가 받는 도구 스키마:")
print(_json.dumps(schema, ensure_ascii=False, indent=2))

📋 AI가 받는 도구 스키마:
{
  "description": "직원 정보를 DB에서 조회합니다.\n\n사용 시점: 특정 직원의 이름, 부서, 직급 정보가 필요할 때.\n주의: 직원 ID는 반드시 'EMP' + 3자리 숫자 형식 (예: EMP001)\n\nArgs:\n    employee_id: 직원 고유 ID (형식: EMP + 3자리 숫자)\nReturns:\n    직원 정보 dict: name, dept, level 키 포함",
  "properties": {
    "employee_id": {
      "title": "Employee Id",
      "type": "string"
    }
  },
  "required": [
    "employee_id"
  ],
  "title": "get_employee_info",
  "type": "object"
}


## Step 1-②. 한 곳만 바꾸기 — docstring이 스키마 description을 결정한다

📖 강의 연계: 모듈 4-2 "비유 ②: @tool docstring은 API 명세서(OpenAPI Spec)"

아래 셀에서 `사용 시점:` 문장을 더 구체적으로 바꾼 뒤 스키마를 다시 출력해보세요.  
**description이 달라지면 AI의 도구 선택 판단 근거도 달라집니다.**

In [ ]:
# ② 한 곳만 바꾸기 — 사용 시점 설명을 더 구체적으로 바꿔보세요
# TODO(🔰): 아래 ___ 를 채워 '사용 시점:' 설명을 더 구체적으로 수정하세요
#            현재: "특정 직원의 이름, 부서, 직급 정보가 필요할 때."
#            예시: "직원 이름·부서·직급 조회 시. 예: EMP001이 누구야?, 박민준 직급은?"
#            (힌트: 📖 강의 모듈 4-2 비유 ②, 모듈 4-3 우수 docstring 예시 참조)

@tool
def get_employee_info_v2(employee_id: str) -> dict:
    """
    직원 정보를 DB에서 조회합니다.
    사용 시점: 직원 입사 날짜 관련 질문이 들어왔을 때
    Args:
        employee_id: 직원 고유 ID (형식: emp + 3자리 숫자)
    Returns:
        직원 정보 dict: name, dept, level 키 포함
    """
    db = {
        "EMP001": {"name": "김철수", "dept": "AI개발팀",  "level": "팀장", "Start date": "2023-01-01"},
        "EMP002": {"name": "이영희", "dept": "기획팀",    "level": "PM", "Start date": "2023-06-01"},
        "EMP003": {"name": "박민준", "dept": "데이터팀",  "level": "시니어", "Start date": "2023-12-01"},
    }
    return db.get(employee_id, {"error": f"직원 없음: {employee_id}"})

import json as _json
schema_v2 = get_employee_info_v2.args_schema.model_json_schema()
print("📋 변경 후 스키마 description:")
print(_json.dumps(schema_v2, ensure_ascii=False, indent=2))

📋 변경 후 스키마 description:
{
  "description": "직원 정보를 DB에서 조회합니다.\n사용 시점: 직원 입사 날짜 관련 질문이 들어왔을 때",
  "properties": {
    "employee_id": {
      "title": "Employee Id",
      "type": "string"
    }
  },
  "required": [
    "employee_id"
  ],
  "title": "get_employee_info_v2",
  "type": "object"
}


## Step 2. 4단계 루프 직접 구현

📖 강의 연계: 모듈 4-2 "v2: 도구 등록 & 단일 루프 완성"

4단계 흐름: ① 등록 → ② AI 호출(tool_calls JSON 생성) → ③ 우리 코드 실행 → ④ 최종 답변

핵심 관찰: AI는 ②에서 **직접 실행하지 않고** JSON만 만든다는 것을 `tool_calls` 출력으로 확인합니다.

In [44]:
# ① 그대로 실행 — 완전한 4단계 루프
# 📖 강의 연계: 모듈 4-2 "v2: 도구 등록 & 단일 루프 완성"

# ─── ① 도구 등록 ──────────────────────────────────────────────
llm_with_tools = llm.bind_tools([get_employee_info_v2])
print(llm_with_tools)

# ─── ② AI 호출 → tool_calls JSON 생성 (AI가 직접 실행하지 않음!) ──
messages = [HumanMessage(content="EMP003 직원 알려줘")]
ai_msg = llm_with_tools.invoke(messages)

print("=" * 55)
print("🔍 [2단계] AI가 생성한 tool_calls JSON:")
print(f"  name  : {ai_msg.tool_calls[0]['name']}")
print(f"  args  : {ai_msg.tool_calls[0]['args']}")
print(f"  id    : {ai_msg.tool_calls[0]['id']}")
print("  직접 실행 여부: ❌ AI는 JSON만 만들었을 뿐 실행하지 않음")

# ─── ③ AI 메시지 이력 추가 (필수! 빠뜨리면 AI가 맥락 상실) ────────
messages.append(ai_msg)

# ─── ③ 우리 코드가 실제 실행 ─────────────────────────────────────
print("\n🔧 [3단계] 우리 코드가 실행한 결과:")
for tc in ai_msg.tool_calls:
    result = get_employee_info_v2.invoke(tc["args"])   # 실제 함수 호출
    print(f"  결과: {result}")
    messages.append(ToolMessage(
        content=str(result),
        tool_call_id=tc["id"],   # AI의 요청 ID와 반드시 연결!
    ))
    print(tc)

# ─── ④ 결과 전달 → 최종 자연어 답변 ─────────────────────────────
final = llm_with_tools.invoke(messages)
print(f"\n🤖 [4단계] 최종 답변: {final.content}")
# 예상 출력: "EMP001 김철수는 AI개발팀 팀장입니다."

bound=ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.6.2', 'langchain': '1.4.0', 'langchain-openai': '1.6.0'}}, profile={'name': 'GPT-4o mini', 'release_date': '2024-07-18', 'last_updated': '2024-07-18', 'open_weights': False, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True, 'tool_call_streaming': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x0000011BFF38C050>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x0000011BFF38CAD0>, root_client=<openai.OpenAI object at 0x0000011BFE4

In [12]:
ai_msg = llm_with_tools.invoke(messages)
ai_msg

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 69, 'total_tokens': 88, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_1d2403c701', 'id': 'chatcmpl-EMODHV5MRaPswdpC8bdZRiPZEpnTD', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a08901-ebd2-7a03-92fa-3c2e4e2b6723-0', tool_calls=[{'name': 'get_employee_info_v2', 'args': {'employee_id': 'EMP003'}, 'id': 'call_KiuX6BhVdJncKcnUx8lJAcKu', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 69, 'output_tokens': 19, 'total_tokens': 88, 'input_token_de

In [ ]:
final

AIMessage(content='EMP003 직원의 정보는 다음과 같습니다:\n\n- **이름**: 박민준\n- **부서**: 데이터팀\n- **직급**: 시니어\n- **입사 날짜**: 2023-12-01', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 54, 'prompt_tokens': 135, 'total_tokens': 189, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_1d2403c701', 'id': 'chatcmpl-EMOKjwlsAhJ9BlItgCTonDNq7EjBm', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a08908-f853-7471-90cb-342cd2157d54-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 135, 'output_tokens': 54, 'total_tokens': 189, 'input_token_details': {'audio': 0, 'cache_read

In [21]:
print(ai_msg.content)

In [20]:
print(final.content)

EMP003 직원의 정보는 다음과 같습니다:

- **이름**: 박민준
- **부서**: 데이터팀
- **직급**: 시니어
- **입사 날짜**: 2023-12-01


## Step 2-②. 한 곳만 바꾸기 — 다른 직원 ID로 조회해보세요

아래 셀에서 직원 ID를 `EMP002` 또는 `EMP003`으로 바꾼 뒤 실행해보세요.  
AI가 `tool_calls`의 `args`에 어떤 ID를 자동으로 넣는지 관찰합니다.

In [17]:
# ② 한 곳만 바꾸기 — 직원 ID를 바꿔 다른 직원을 조회하세요
# TODO(🔰): 아래 ___ 를 채우세요 (힌트: EMP002 또는 EMP003)

query = "EMP002 직원 정보 알려줘"   # ← 002 또는 003을 채우세요

messages_2 = [HumanMessage(content=query)]
ai_msg_2 = llm_with_tools.invoke(messages_2)

print("AI가 tool_calls에 넣은 args:", ai_msg_2.tool_calls[0]["args"])

messages_2.append(ai_msg_2)
for tc in ai_msg_2.tool_calls:
    result = get_employee_info.invoke(tc["args"])
    messages_2.append(ToolMessage(content=str(result), tool_call_id=tc["id"]))

final_2 = llm_with_tools.invoke(messages_2)
print("최종 답변:", final_2.content)

AI가 tool_calls에 넣은 args: {'employee_id': 'EMP002'}
최종 답변: EMP002 직원의 정보는 다음과 같습니다:

- 이름: 이영희
- 부서: 기획팀
- 직급: PM (프로젝트 매니저)


## Step 3. 분기 처리 — tool_calls가 없을 때 (AI 직접 답변)

📖 강의 연계: 모듈 4-2 "분기 처리: tool_calls가 없을 때 (AI가 직접 답변)"

AI가 도구를 쓰지 않고 직접 답변하는 경우가 있습니다.  
`tool_calls`가 빈 리스트(`[]`)일 때와 아닐 때를 반드시 분기 처리해야 합니다.

> ⚠️ **흔한 실수 3번**: `ai_msg.tool_calls`가 `[]`인데 `tool_calls[0]`에 접근하면 `IndexError` 발생!

In [23]:
# ① 그대로 실행 — 분기 처리 완성 코드
# 📖 강의 연계: 모듈 4-2 "분기 처리: tool_calls가 없을 때"

def run_with_branch(question: str) -> str:
    """도구 호출 유무에 따라 분기 처리하는 메인 함수"""
    messages = [HumanMessage(content=question)]
    ai_msg = llm_with_tools.invoke(messages)

    if ai_msg.tool_calls:                    # 도구 호출이 있는 경우
        messages.append(ai_msg)
        for tc in ai_msg.tool_calls:
            result = get_employee_info.invoke(tc["args"])
            messages.append(ToolMessage(
                content=str(result), tool_call_id=tc["id"],
            ))
        final = llm_with_tools.invoke(messages)
        return f"[도구 사용] {final.content}"
    else:                                    # 직접 답변하는 경우
        return f"[직접 답변] {ai_msg.content}"

# 두 경로를 모두 테스트
print(run_with_branch("EMP001 직원 정보 알려줘"))  # → 도구 사용 경로
print()
print(run_with_branch("안녕! 오늘 날씨 좋네"))     # → 직접 답변 경로

[도구 사용] EMP001 직원의 정보는 다음과 같습니다:

- 이름: 김철수
- 부서: AI개발팀
- 직급: 팀장

[직접 답변] 안녕하세요! 날씨가 좋다니 기쁘네요. 오늘 어떤 계획이 있으신가요?


## Step 3-②. 한 곳만 바꾸기 — 나만의 질문으로 두 경로를 테스트하세요

도구를 사용하는 질문과 도구 없이 AI가 직접 답변하는 질문을 각각 1개씩 만들어보세요.

In [24]:
# ② 한 곳만 바꾸기 — 나만의 질문으로 두 경로를 확인하세요
# TODO(🔰): 아래 ___ 를 채우세요
#   tool_question: 직원 정보를 조회해야 하는 질문 (도구 사용 경로)
#   chat_question: 일상 대화 질문 (직접 답변 경로)
#   (힌트: 📖 강의 모듈 4-2 분기 처리 예시 참조)

tool_question = "EMP003에 대해 알려줘"   # 예: "EMP003은 누구야?"
chat_question = "LLM에서 활용하는 tool에 대해 설명해줘"   # 예: "파이썬이 뭐야?"

print("--- 도구 사용 경로 ---")
print(run_with_branch(tool_question))
print()
print("--- 직접 답변 경로 ---")
print(run_with_branch(chat_question))

--- 도구 사용 경로 ---
[도구 사용] EMP003에 대한 정보는 다음과 같습니다:

- **이름**: 박민준
- **부서**: 데이터팀
- **직급**: 시니어

--- 직접 답변 경로 ---
[직접 답변] LLM(대형 언어 모델)에서 활용하는 도구는 주로 특정 작업을 수행하거나 정보를 조회하기 위해 사용됩니다. 이러한 도구들은 LLM의 기능을 확장하고, 더 정확하고 유용한 결과를 제공하는 데 도움을 줍니다. 예를 들어, 데이터베이스에서 정보를 조회하거나, 특정 계산을 수행하거나, 외부 API와 상호작용하는 등의 작업을 수행할 수 있습니다.

도구의 주요 기능은 다음과 같습니다:

1. **정보 조회**: 특정 데이터베이스나 시스템에서 정보를 검색하여 사용자에게 제공할 수 있습니다. 예를 들어, 직원의 입사 날짜나 특정 제품의 재고 상태를 조회할 수 있습니다.

2. **데이터 처리**: 수치 계산, 데이터 변환, 텍스트 분석 등 다양한 데이터 처리 작업을 수행할 수 있습니다.

3. **API 통합**: 외부 서비스와 통신하여 필요한 정보를 가져오거나, 특정 작업을 수행할 수 있습니다. 예를 들어, 날씨 정보를 가져오거나, 결제 처리를 할 수 있습니다.

4. **자동화**: 반복적인 작업을 자동으로 수행하여 사용자의 시간을 절약할 수 있습니다.

이러한 도구들은 LLM이 더 많은 정보를 처리하고, 사용자에게 더 나은 서비스를 제공할 수 있도록 돕는 중요한 역할을 합니다.


## Step 4. 다중 도구 & description 품질 실험

📖 강의 연계: 모듈 4-3 "여러 도구 중 AI가 선택" / "Description 품질 실험 — ❌ vs ✅"

여러 도구를 등록하면 AI가 질문의 의도에 따라 도구를 선택합니다.  
**도구 선택 정확도는 전적으로 docstring description 품질에 달려 있습니다.**

> 💡 이 실험 결과가 10월 MCP 도구 설계의 핵심 원칙이 됩니다.

In [13]:
# ① 그대로 실행 — 3개 도구 등록 + 질문에 따른 선택 관찰
# 📖 강의 연계: 모듈 4-3 "여러 도구 중 AI가 선택하는 방법"

@tool
def calculate(expression: str) -> str:
    """
    수학 계산을 수행합니다.
    사용 시점: 사칙연산 등 숫자 계산이 필요할 때.
    expression: 계산 가능한 수식 (예: '1234 * 567', '100 / 4')
    """
    # ⚠️ 교육용 코드: eval()은 프로덕션에서 사용 금지 (보안 취약점)
    try: return str(eval(expression))
    except Exception as e: return f"계산 오류: {str(e)}"

@tool
def get_weather(city: str) -> str:
    """
    도시 날씨를 조회합니다.
    사용 시점: 특정 도시의 현재 날씨·기온을 물어볼 때.
    예시 질문: '서울 날씨', '오늘 도쿄 기온', '부산 날씨 알려줘'
    """
    return f"{city}: 맑음, 25도"  # Mock 데이터

# 3개 동시 등록
llm_multi = llm.bind_tools([get_employee_info, calculate, get_weather])
test_questions = [
    "서울 날씨 알려줘",          # 예상 선택: get_weather
    "1234 * 567은 얼마야?",     # 예상 선택: calculate
    "EMP002 직원 정보 알려줘",   # 예상 선택: get_employee_info
    "안녕!",                    # 예상: 도구 없음 (직접 답변)
]

print("질문별 도구 선택 결과:")
print("=" * 55)
for q in test_questions:
    messages = [HumanMessage(content=q)]
    msg = llm_multi.invoke(messages)
    messages.append(msg)
    if msg.tool_calls:
        tc = msg.tool_calls[0] #길이 1짜리 리스트인데, 이걸 하면 tc는 dictionary가 됨
        if tc['name'] == 'get_employee_info':
            result = get_employee_info.invoke(tc["args"])
            messages.append(ToolMessage(content = str(result), tool_call_id = tc['id']))
            final = llm_multi.invoke(messages)
            print(f"Q: {q}")
            print(final.content)
            # print(f"   → ✅ 선택된 도구: get_employee_info")
            # print(f"      인자: {tc['args']}")
        elif tc['name'] == 'calculate':
            result = calculate.invoke(tc["args"])
            messages.append(ToolMessage(content = str(result), tool_call_id = tc['id']))
            final = llm_multi.invoke(messages)
            print(f"Q: {q}")
            print(final.content)
            # print(f"   → ✅ 선택된 도구: calculate")
            # print(f"      인자: {tc['args']}")
        elif tc['name'] == 'get_weather':
            result = get_weather.invoke(tc["args"])
            messages.append(ToolMessage(content = str(result), tool_call_id = tc['id']))
            final = llm_multi.invoke(messages)
            print(f"Q: {q}")
            print(final.content)
            # print(f"   → ✅ 선택된 도구: get_weather")
            # print(f"      인자: {tc['args']}")
        else:
            pass
    else:
        print(f"Q: {q}")
        print(f"   → 💬 직접 답변: {msg.content[:40]}...")



질문별 도구 선택 결과:
Q: 서울 날씨 알려줘
서울의 현재 날씨는 맑고, 기온은 25도입니다.
Q: 1234 * 567은 얼마야?
1234 * 567은 699,678입니다.
Q: EMP002 직원 정보 알려줘
EMP002 직원 정보는 다음과 같습니다:

- 이름: 이영희
- 부서: 기획팀
- 직급: PM (프로젝트 매니저)
Q: 안녕!
   → 💬 직접 답변: 안녕하세요! 어떻게 도와드릴까요?...


In [ ]:
#위의 조건문을 dic 활용하여 좀 더 개량함
@tool
def calculate(expression: str) -> str:
    """
    수학 계산을 수행합니다.
    사용 시점: 사칙연산 등 숫자 계산이 필요할 때.
    expression: 계산 가능한 수식 (예: '1234 * 567', '100 / 4')
    """
    # ⚠️ 교육용 코드: eval()은 프로덕션에서 사용 금지 (보안 취약점)
    try: return str(eval(expression))
    except Exception as e: return f"계산 오류: {str(e)}"

@tool
def get_weather(city: str) -> str:
    """
    도시 날씨를 조회합니다.
    사용 시점: 특정 도시의 현재 날씨·기온을 물어볼 때.
    예시 질문: '서울 날씨', '오늘 도쿄 기온', '부산 날씨 알려줘'
    """
    return f"{city}: 맑음, 25도"  # Mock 데이터

# 3개 동시 등록
llm_multi = llm.bind_tools([get_employee_info, calculate, get_weather])
test_questions = [
    "서울 날씨 알려줘",          # 예상 선택: get_weather
    "1234 * 567은 얼마야?",     # 예상 선택: calculate
    "EMP002 직원 정보 알려줘",   # 예상 선택: get_employee_info
    "안녕!",                    # 예상: 도구 없음 (직접 답변)
]
dic = {'get_employee_info': get_employee_info, 'calculate': calculate, 'get_weather': get_weather}
print("질문별 도구 선택 결과:")
print("=" * 55)
for q in test_questions:
    messages = [HumanMessage(content=q)]
    msg = llm_multi.invoke(messages)
    messages.append(msg)
    if msg.tool_calls:
        tc = msg.tool_calls[0] #길이 1짜리 리스트인데, 이걸 하면 tc는 dictionary가 됨
        result = dic[tc['name']].invoke(tc["args"])
        messages.append(ToolMessage(content = str(result), tool_call_id = tc['id']))
        final = llm_multi.invoke(messages)
        print(f"Q: {q}")
        print(final.content)
    else:
        print(f"Q: {q}")
        print(f"   → 💬 직접 답변: {msg.content[:40]}...")



질문별 도구 선택 결과:
Q: 서울 날씨 알려줘
서울의 현재 날씨는 맑고, 기온은 25도입니다.
Q: 1234 * 567은 얼마야?
1234 * 567은 699,678입니다.
Q: EMP002 직원 정보 알려줘
EMP002 직원 정보는 다음과 같습니다:

- 이름: 이영희
- 부서: 기획팀
- 직급: PM (프로젝트 매니저)
Q: 안녕!
   → 💬 직접 답변: 안녕하세요! 어떻게 도와드릴까요?...


In [ ]:
#강사 버전
tools = [get_employee_info, calculate, get_weather]
tools_map = {t.name: t for t in tools}    #이렇게 딕셔너리를 설정하면 내가 tools에 도구만 넣어도 딕셔너리에 자동으로 추가된다
llm_multi = llm.bind_tools(tools)

print("질문별 도구 선택 결과:")
print("=" * 55)
for q in test_questions:
    messages = [HumanMessage(content=q)]
    msg = llm_multi.invoke(messages)
    messages.append(msg)
    if msg.tool_calls:
        tc = msg.tool_calls[0] #길이 1짜리 리스트인데, 이걸 하면 tc는 dictionary가 됨
        result = tools_map[tc['name']].invoke(tc["args"])
        messages.append(ToolMessage(content = str(result), tool_call_id = tc['id']))
        final = llm_multi.invoke(messages)
        print(f"Q: {q}")
        print(final.content)
    else:
        print(f"Q: {q}")
        print(f"   → 💬 직접 답변: {msg.content[:40]}...")


질문별 도구 선택 결과:
Q: 서울 날씨 알려줘
서울의 현재 날씨는 맑고, 기온은 25도입니다.
Q: 1234 * 567은 얼마야?
1234 * 567은 699,678입니다.
Q: EMP002 직원 정보 알려줘
EMP002 직원 정보는 다음과 같습니다:

- 이름: 이영희
- 부서: 기획팀
- 직급: PM (프로젝트 매니저)
Q: 안녕!
   → 💬 직접 답변: 안녕하세요! 어떻게 도와드릴까요?...


## Step 4-②. 한 곳만 바꾸기 — description을 짧게 줄여 선택 실패를 관찰하세요

📖 강의 연계: 모듈 4-3 "Description 품질 실험 — ❌ vs ✅"

아래 셀에서 `get_weather_short`의 docstring을 매우 짧게 (예: `날씨.`) 줄여보세요.  
같은 질문에 어떤 도구가 선택되는지 ❌ vs ✅ 결과를 비교합니다.

> ⚠️ description 품질 = 도구 선택 정확도

In [ ]:
# ② 한 곳만 바꾸기 — 짧은 description이 어떤 문제를 일으키는지 관찰
# TODO(🔰): 아래 ___ 를 채우세요
#   get_weather_short의 docstring을 아주 짧게 ('날씨.' 처럼) 채우세요
#   (힌트: 📖 강의 모듈 4-3 '❌ 나쁜 예' 참조)

@tool
def get_weather_short(city: str) -> str:
    """날씨"""   # ← 여기를 "날씨." 처럼 짧게 채우세요
    return f"{city}: 맑음, 25도"

llm_bad = llm.bind_tools([get_employee_info, calculate, get_weather_short])

print("❌ 짧은 description — '도쿄 내일 날씨 알려줘' 질문:")
msg_bad = llm_bad.invoke([HumanMessage(content="도쿄 내일 날씨 알려줘")])
if msg_bad.tool_calls:
    print(f"   선택된 도구: {msg_bad.tool_calls[0]['name']}")
else:
    print(f"   도구 없음 — 직접 답변: {msg_bad.content[:40]}...")

print("\n✅ 충분한 description — 같은 질문:")
msg_good = llm_multi.invoke([HumanMessage(content="도쿄 내일 날씨 알려줘")])
if msg_good.tool_calls:
    print(f"   선택된 도구: {msg_good.tool_calls[0]['name']}")

print("\n👉 두 결과를 비교해 아래 메모 셀에 이유를 적어보세요")

❌ 짧은 description — '도쿄 내일 날씨 알려줘' 질문:
   선택된 도구: get_weather_short

✅ 충분한 description — 같은 질문:
   선택된 도구: get_weather

👉 두 결과를 비교해 아래 메모 셀에 이유를 적어보세요


In [ ]:
# 👀 관찰 메모 — 실험 결과를 기록하세요 (코드 실행 불필요)

# 짧은 description일 때 선택된 도구: get_weather_short
# 충분한 description일 때 선택된 도구: get_weather
# 왜 차이가 생겼는가 (나의 해석 1줄): description 길이와 상관없이 위 결과는 weather 관련 도구가 llm_bad에는 get_weather_short만 있고, llm_multi에는 get_weather만 있기 때문에 발생함
#

In [30]:
# 이번에는 llm_weather_2만 써서 weather를 2개 넣었을 때 어디를 선택하는지 확인해보자

@tool
def get_weather_short(city: str) -> str:
    """날씨"""   # ← 여기를 "날씨." 처럼 짧게 채우세요
    return f"{city}: 맑음, 25도"

llm_weather_2 = llm.bind_tools([get_employee_info, calculate, get_weather, get_weather_short])

print("get_weather_short에 날씨만 적은 상태")
print("도쿄 내일 날씨 알려줘 질문에 둘 중 어떤 것 고르는지 확인")
msg = llm_weather_2.invoke([HumanMessage(content="도쿄 내일 날씨 알려줘")])
if msg.tool_calls:
    print(f"   선택된 도구: {msg.tool_calls[0]['name']}")
else:
    print(f"   도구 없음 — 직접 답변: {msg.content[:40]}...")

print("내일 날씨 질문에 둘 중 어떤 것 고르는지 확인")
msg = llm_weather_2.invoke([HumanMessage(content="내일 날씨")])
if msg.tool_calls:
    print(f"   선택된 도구: {msg.tool_calls[0]['name']}")
else:
    print(f"   도구 없음 — 직접 답변: {msg.content[:40]}...")

print("날씨 질문에 둘 중 어떤 것 고르는지 확인")
msg = llm_weather_2.invoke([HumanMessage(content="날씨")])
if msg.tool_calls:
    print(f"   선택된 도구: {msg.tool_calls[0]['name']}")
else:
    print(f"   도구 없음 — 직접 답변: {msg.content[:40]}...")

print("결론: description을 대충 작성하거나 HumanMessage를 대충 작성하면 인식을 잘 못함")

get_weather_short에 날씨만 적은 상태
도쿄 내일 날씨 알려줘 질문에 둘 중 어떤 것 고르는지 확인
   선택된 도구: get_weather
내일 날씨 질문에 둘 중 어떤 것 고르는지 확인
   도구 없음 — 직접 답변: 어떤 도시의 내일 날씨를 알고 싶으신가요?...
날씨 질문에 둘 중 어떤 것 고르는지 확인
   도구 없음 — 직접 답변: 어떤 도시의 날씨를 알고 싶으신가요? 도시 이름을 말씀해 주시면 그에 대...
결론: description을 대충 작성하거나 HumanMessage를 대충 작성하면 인식을 잘 못함


## ⭐ Step 4 심화 실습 — 도구 선택 제어 & 다국어 실험

📖 강의 연계: 모듈 4-3 "⭐ 심화 실습 1·2번"

기본 미션을 완료한 뒤 도전하세요. 힌트 없이 직접 구현합니다.

**과제 1**: `tool_choice` 옵션으로 AI가 항상 `calculate`를 쓰도록 강제해보세요.
**과제 2**: docstring을 영어로만 쓰면 한국어 질문에서도 올바른 도구가 선택될까요?

In [ ]:
# ⭐ Step 4 심화 1 — 도구 선택 강제 (tool_choice)
# 📖 강의 연계: 모듈 4-3 "⭐ 심화 실습 1번"
# tool_choice 옵션을 사용하면 AI가 특정 도구만 선택하도록 강제할 수 있습니다.
# TODO(⭐): llm.bind_tools(...)에 tool_choice 옵션을 추가해 calculate만 강제하세요.
#   힌트: tool_choice={"type": "function", "function": {"name": "calculate"}}
#   확인: "서울 날씨 알려줘" 같은 날씨 질문도 calculate가 선택되는지 관찰

# 강제 없이 (기본) — 날씨 질문에 get_weather 선택됨
q_weather = "서울 날씨 알려줘"
normal = llm_multi.invoke([HumanMessage(content=q_weather)])
print("기본 (tool_choice 없음):")
print(f"  선택된 도구: {normal.tool_calls[0]['name'] if normal.tool_calls else '직접 답변'}")

# TODO(⭐): 아래 ___ 를 채워 calculate만 강제하세요
llm_forced = llm.bind_tools(
    [get_employee_info, calculate, get_weather],
    tool_choice="___",   # ← {"type": "function", "function": {"name": "calculate"}} 형식으로 채우기
)

forced = llm_forced.invoke([HumanMessage(content=q_weather)])
print("\n강제 (tool_choice=calculate):")
print(f"  선택된 도구: {forced.tool_calls[0]['name'] if forced.tool_calls else '직접 답변'}")
print("→ 날씨 질문인데도 calculate가 선택됐나요? 이것이 tool_choice의 효과입니다.")

In [ ]:
# ⭐ Step 4 심화 2 — 다국어 description 실험
# 📖 강의 연계: 모듈 4-3 "⭐ 심화 실습 2번"
# docstring을 영어로만 쓰면 한국어 질문에서도 올바른 도구가 선택될까요?

# 영어 docstring 버전의 날씨 도구 정의
@tool
def get_weather_en(city: str) -> str:
    """
    Get the current weather for a city.
    Use when: user asks about weather, temperature, or climate of a specific city.
    """
    return f"{city}: clear, 25 degrees C"

llm_en = llm.bind_tools([get_employee_info, calculate, get_weather_en])

test_lang = [
    "서울 날씨 알려줘",       # 한국어 질문 → 영어 docstring 도구
    "도쿄 기온이 어때?",      # 한국어 질문 → 영어 docstring 도구
    "1234 * 567은?",         # 계산 질문
]

print("영어 docstring 도구 — 한국어 질문 선택 결과:")
print("=" * 50)
for q in test_lang:
    msg = llm_en.invoke([HumanMessage(content=q)])
    chosen = msg.tool_calls[0]["name"] if msg.tool_calls else "직접 답변"
    print(f"Q: {q}")
    print(f"   → {chosen}")
    print()

# 👀 관찰 메모:
# 영어 docstring이어도 한국어 질문에서 올바른 도구가 선택됐나요?
# LangSmith 트레이스에서 AI가 영어 description을 어떻게 해석했는지 확인하세요.
# 결과: ___

---

## Step 5. 실제 외부 API 도구 — 웹 검색·뉴스

📖 강의 연계: 모듈 4-3 "⭐ 실제 외부 API 도구 — 웹 검색·뉴스·논문"

Step 1~4의 도구는 모두 **Mock 데이터**를 반환했습니다.  
이번 Step에서는 **Tavily API**를 이용해 실제 인터넷에서 정보를 가져오는 도구를 만듭니다.

**LLM의 학습 데이터 컷오프를 도구로 극복하는 가장 전형적인 패턴입니다.**

> ⚠️ 이 Step은 TAVILY_API_KEY가 필요합니다.
> 발급: [app.tavily.com](https://app.tavily.com) → 무료 계정 생성 → API Keys
> `.env`에 `TAVILY_API_KEY=tvly-xxxx` 추가 후 커널 재시작

키가 없어도 셀은 fallback 모드로 실행됩니다 — 구조와 흐름을 먼저 이해하세요.

In [4]:
# Step 5 사전 확인 — Tavily API 키 상태
# 📖 강의 연계: 모듈 4-3 "사전 준비 — Tavily API 키 발급 & .env 추가"

# TAVILY_AVAILABLE 은 셀[01]에서 설정됨
if TAVILY_AVAILABLE:
    import os
    key = os.getenv("TAVILY_API_KEY")
    print(f"✅ TAVILY_API_KEY 확인: {key[:7]}...")
    print("   pip install langchain_tavily tavily-python  ← 미설치 시 실행")
else:
    print("⚠️  TAVILY_API_KEY 없음")
    print("   발급: https://app.tavily.com → Create API Key")
    print("   .env 에 추가: TAVILY_API_KEY=tvly-xxxx")
    print("   추가 후 커널 재시작 → 셀[01] 재실행")
    print()
    print("아래 셀은 fallback 모드(Mock 결과)로 실행됩니다.")

print(f"\n📦 Tavily 실습 가능: {'✅' if TAVILY_AVAILABLE else '❌ (fallback 모드)'}")

✅ TAVILY_API_KEY 확인: tvly-de...
   pip install langchain_tavily tavily-python  ← 미설치 시 실행

📦 Tavily 실습 가능: ✅


In [8]:
# ① 그대로 실행 — TavilySearch raw 결과 확인
# 📖 강의 연계: 모듈 4-3 "v1: TavilySearch 기본 사용 — raw 결과 확인"

if TAVILY_AVAILABLE:
    from langchain_tavily import TavilySearch
    tavily = TavilySearch(max_results=3)
    raw = tavily.invoke("봉누도2 기자 명단")
    results = raw.get("results", [])
    print(f"총 결과 수: {len(results)}개")
    if results:
        r0 = results[0]
        print(f"  title  : {r0.get('title', 'N/A')}")
        print(f"  url    : {r0.get('url', 'N/A')}")
        print(f"  content: {r0.get('content', '')[:150]}...")
        print(f"  score  : {r0.get('score', 'N/A')}")
    print()
    print("💡 관찰: 불필요한 필드(raw_content 등)가 많음 → @tool로 포맷 정제 필요")
else:
    print("[fallback] 예상 출력:")
    print("  총 결과 수: 3개")
    print("  title  : LangChain v0.3 Release Notes")
    print("  url    : https://python.langchain.com/...")
    print("  content: LangChain 0.3 introduces ...")
    print("  score  : 0.98")
    print("  → raw_content, metadata 등 불필요한 필드도 포함됨")

총 결과 수: 3개
  title  : 봉누도 2/집단 및 세력/봉누도방송국 - 나무위키
  url    : https://namu.wiki/w/%EB%B4%89%EB%88%84%EB%8F%84%202/%EC%A7%91%EB%8B%A8%20%EB%B0%8F%20%EC%84%B8%EB%A0%A5/%EB%B4%89%EB%88%84%EB%8F%84%EB%B0%A9%EC%86%A1%EA%B5%AD
  content: 소속 인원 (10명). 보도국장. 이윤진 이춘향 ; 소속 인원 (10명) · 부국장. 미정 미정 ; 소속 인원 (10명) · 기자. 미정 로마러. 미정 마레 플로스. 미정 시라유키 히나. 미정...
  score  : 0.79119295

💡 관찰: 불필요한 필드(raw_content 등)가 많음 → @tool로 포맷 정제 필요


In [9]:
results

[{'url': 'https://namu.wiki/w/%EB%B4%89%EB%88%84%EB%8F%84%202/%EC%A7%91%EB%8B%A8%20%EB%B0%8F%20%EC%84%B8%EB%A0%A5/%EB%B4%89%EB%88%84%EB%8F%84%EB%B0%A9%EC%86%A1%EA%B5%AD',
  'title': '봉누도 2/집단 및 세력/봉누도방송국 - 나무위키',
  'content': '소속 인원 (10명). 보도국장. 이윤진 이춘향 ; 소속 인원 (10명) · 부국장. 미정 미정 ; 소속 인원 (10명) · 기자. 미정 로마러. 미정 마레 플로스. 미정 시라유키 히나. 미정',
  'score': 0.79119295,
  'raw_content': None,
  'id': '3e663d-00'},
 {'url': 'https://arca.live/b/stellive/182496315',
  'title': '봉누도2 기자 합격자 명단 - 스텔라이브 채널',
  'content': '정보 봉누도2 기자 합격자 명단. 히리느. 추천 2 비추천 0 댓글 11 ... 92909 정보 봉누도2 기자 합격자 명단 [11]. 히리느 04:00 1754 2. 92908',
  'score': 0.67269784,
  'raw_content': None,
  'id': '166c48-01'},
 {'url': 'https://www.inven.co.kr/board/party/6387/2748',
  'title': '정보 봉누도 2 EMS 합격자 발표 - 인벤',
  'content': '정보 봉누도 2 기자 합격자 발표 사진 회원 아이콘 이미지 [Cheatkey] 조회 2937 20:03 6 댓글',
  'score': 0.33816507,
  'raw_content': None,
  'id': '137188-02'}]

## Step 5-②. 한 곳만 바꾸기 — @tool로 Tavily 결과 포맷 정제하기

📖 강의 연계: 모듈 4-3 "v2: @tool로 결과 포맷 다듬기 — 웹 검색"

raw 결과에는 AI에게 불필요한 필드가 많습니다. `@tool`로 래핑해 필요한 정보만 전달합니다.  
**`docstring`의 "사용 시점:"이 AI의 도구 선택 판단 근거가 됩니다.**

TODO: `web_search` 함수의 `사용 시점:` 과 `사용하지 말 것:` 두 줄을 채우세요.

In [10]:
# ② 한 곳만 바꾸기 — @tool 래핑으로 결과 포맷 정제
# 📖 강의 연계: 모듈 4-3 "v2: @tool로 결과 포맷 다듬기 — 웹 검색"
# TODO(⭐): 아래 ___ 를 채워 docstring "사용 시점:"을 작성하세요
#           (힌트: 언제 써야 하나? / 역사적 사실처럼 쓰지 말아야 할 때는?)

from langchain_core.tools import tool as _tool

if TAVILY_AVAILABLE:
    from langchain_tavily import TavilySearch as _TS

    @_tool
    def web_search(query: str, max_results: int = 5) -> str:
        """
        웹을 실시간으로 검색해 최신 정보를 가져옵니다.
        사용 시점: 기존 모델이 학습하지 않은 정보가 입력으로 들어왔을 때
        사용하지 말 것: 개인정보

        Args:
            query: 검색어 (한국어/영어 모두 가능)
            max_results: 반환할 결과 수 (기본 5, 최대 20)
        """
        results = _TS(max_results=max_results).invoke(query)["results"]
        return "\n---\n".join(
            f"제목: {d.get('title','N/A')}\nURL: {d.get('url','')}\n내용: {d.get('content','')}"
            for d in results
        )

    import json as _j
    print("📋 web_search 도구 스키마:")
    print(_j.dumps(web_search.args_schema.model_json_schema(), ensure_ascii=False, indent=2))

else:
    # fallback — 키 없어도 이후 셀이 돌아가도록 도구 정의
    @_tool
    def web_search(query: str, max_results: int = 5) -> str:
        """
        웹을 실시간으로 검색해 최신 정보를 가져옵니다.
        사용 시점: LLM 학습 데이터 이후 최신 정보가 필요할 때.
        사용하지 말 것: 역사적 사실, 일반 상식.
        """
        return f"[fallback] '{query}' 검색 — Tavily 키 설정 후 실제 결과 확인"

    print("✅ web_search 정의 완료 (fallback 모드)")
    print("   TODO 힌트: 사용 시점 = LLM이 모르는 최신 정보 / 사용 안 할 때 = 알고 있는 사실")

📋 web_search 도구 스키마:
{
  "description": "웹을 실시간으로 검색해 최신 정보를 가져옵니다.\n사용 시점: 기존 모델이 학습하지 않은 정보가 입력으로 들어왔을 때\n사용하지 말 것: 개인정보\n\nArgs:\n    query: 검색어 (한국어/영어 모두 가능)\n    max_results: 반환할 결과 수 (기본 5, 최대 20)",
  "properties": {
    "query": {
      "title": "Query",
      "type": "string"
    },
    "max_results": {
      "default": 5,
      "title": "Max Results",
      "type": "integer"
    }
  },
  "required": [
    "query"
  ],
  "title": "web_search",
  "type": "object"
}


## Step 5-③. 내 것에 적용 — 뉴스 검색 도구 + 4도구 조합 실행

📖 강의 연계: 모듈 4-3 "v3: 뉴스 검색 — topic 파라미터" / "Mock + 실제 외부 API 도구 조합 실행"

`topic="news"`를 사용하면 뉴스 기사만 특화해 검색합니다.  
웹 검색과 **별도 도구**로 분리하면 AI가 질문 의도에 따라 더 정확하게 선택합니다.

**4개 도구(직원조회 + 계산 + 웹검색 + 뉴스)를 동시에 등록하고 5가지 질문으로 선택 결과를 관찰합니다.**

TODO: `news_search` 함수의 `사용 시점:` 을 채우고, 내용 앞 몇 자를 남길지 `N`을 정하세요.

In [14]:
# ③ 내 것에 적용 — news_search 완성 + 4도구 조합 실행
# 📖 강의 연계: 모듈 4-3 "v3: 뉴스 검색" / "Mock + 실제 외부 API 도구 조합 실행"
# TODO(⭐): 아래 ___ 를 채우세요
#   1. news_search docstring "사용 시점:" 작성
#   2. content 앞 N자 제한: content[:N]에서 적절한 N 결정 (뉴스는 짧게)

if TAVILY_AVAILABLE:
    from langchain_tavily import TavilySearch as _TS2

    @_tool
    def news_search(query: str, max_results: int = 5) -> str:
        """
        최신 뉴스 기사를 검색합니다.

        사용 시점: 뉴스와 관련된 입력이 들어왔을 때
        사용하지 말 것: 역사적 사실이나 웹 일반 검색으로 충분한 질문.

        Args:
            query: 뉴스 검색어
            max_results: 결과 수 (기본 5)
        """
        results = _TS2(max_results=max_results, topic="news").invoke(query)["results"]
        return "\n---\n".join(
            f"제목: {d.get('title','N/A')}\nURL: {d.get('url','')}\n내용: {d.get('content','')[:___]}"
            for d in results
        )

else:
    @_tool
    def news_search(query: str, max_results: int = 5) -> str:
        """
        최신 뉴스 기사를 검색합니다.
        사용 시점: 오늘 뉴스, 최근 사건·사고·정책 발표 등 시사 정보가 필요할 때.
        """
        return f"[fallback] '{query}' 뉴스 — Tavily 키 설정 후 실제 결과 확인"

# ── 4도구 동시 등록 + 5개 질문으로 선택 실험 ──────────────────────────
llm_full = llm.bind_tools([get_employee_info, calculate, web_search, news_search])

test_5 = [
    "GPT-5는 언제 출시돼?",          # 예상: web_search  (최신 정보)
    "오늘 AI 관련 주요 뉴스 알려줘",   # 예상: news_search (시사)
    "EMP003 직원 정보 알려줘",         # 예상: get_employee_info
    "256 * 1024 계산해줘",            # 예상: calculate
    "파이썬이 뭐야?",                  # 예상: 직접 답변
]

print("질문별 도구 선택 결과 (4개 도구):")
print("=" * 60)
for q in test_5:
    msg = llm_full.invoke([HumanMessage(content=q)])
    chosen = msg.tool_calls[0]["name"] if msg.tool_calls else "직접 답변"
    print(f"Q: {q}")
    print(f"   → {chosen}")
    print()

질문별 도구 선택 결과 (4개 도구):
Q: GPT-5는 언제 출시돼?
   → web_search

Q: 오늘 AI 관련 주요 뉴스 알려줘
   → news_search

Q: EMP003 직원 정보 알려줘
   → get_employee_info

Q: 256 * 1024 계산해줘
   → calculate

Q: 파이썬이 뭐야?
   → 직접 답변



In [ ]:
# 👀 Step 5 관찰 메모 (코드 실행 불필요)
# 📖 강의 연계: 모듈 4-3 "⭐ 실제 외부 API 도구 — 웹 검색·뉴스·논문"

# web_search vs news_search 선택 구분이 잘 됐나요?
# 관찰 결과: ___

# LangSmith 트레이스에서 Latency를 비교해보세요
# Mock 도구(get_employee_info) latency : ___ ms
# Tavily 도구(web_search)      latency : ___ ms

# (⭐) docstring "사용 시점:"을 바꾸면 선택 결과가 달라졌나요?
# 실험 내용: ___

---

## 🎯 연습 문제 — 8교시 실습 내용 적용

📖 강의 연계: 모듈 4-1~4-3 전체

아래 4개 연습 문제는 `8교시_LLM_Agent와_Tool_Calling.ipynb` 실습 내용을 기반으로 합니다.  
Step 1~5에서 배운 개념을 다른 각도로 확인합니다.

In [ ]:
# 🎯 연습 1 — bind_tools가 AI에게 전달하는 JSON 스키마를 직접 확인하세요
# 📖 강의 연계: 모듈 4-1 "tool_calls 이해" / 모듈 4-2 "Pydantic과의 연결"
#
# LangChain은 bind_tools() 호출 시 각 @tool의 정보를
# JSON Schema 형식으로 변환해 AI의 시스템 프롬프트에 포함합니다.
# AI는 이 JSON만 보고 도구 선택 판단을 합니다.

import json as _j

# llm_multi 는 Step 4에서 [get_employee_info, calculate, get_weather] 로 bind됨
tools_json = llm_multi.kwargs["tools"]

print("📋 AI가 받는 전체 도구 스키마 (JSON):")
print(_j.dumps(tools_json, ensure_ascii=False, indent=2))

print("\n💡 관찰 포인트:")
print(f"  ① 등록된 도구 수: {len(tools_json)}개")
print(f"  ② 첫 번째 도구 이름: {tools_json[0]['function']['name']}")
print(f"  ③ description 위치: tools[N]['function']['description']")
print()
print("→ docstring이 여기 'description'으로 들어갑니다.")
print("  AI는 이 description만 읽고 언제 이 도구를 쓸지 판단합니다.")

## 🎯 연습 2 — "LLM은 오늘 날짜를 모릅니다" → current_date 도구로 해결

📖 강의 연계: 모듈 4-2 "왜 배우는가" / 모듈 4-3 "실제 외부 API 도구"

LLM의 학습 데이터에는 미래 정보가 없습니다. "오늘 날짜"도 마찬가지입니다.  
이것이 도구 연결이 필요한 **가장 단순하고 명확한 이유**입니다.

① 먼저 도구 없이 날짜를 물어봐 LLM의 한계를 확인합니다.  
② `current_date` 도구를 정의하고 연결해 정확한 답을 얻습니다.

In [15]:
# 🎯 연습 2 — LLM이 날짜를 모른다 확인 + current_date 도구로 해결
# 📖 강의 연계: 모듈 4-2 "왜 배우는가"

from datetime import datetime

# ─── A: 도구 없이 날짜 질문 ─────────────────────────────────────────
print("=== A: 도구 없이 질문 (LLM이 추측합니다) ===")
no_tool = llm.invoke("오늘 날짜는?")
print(f"LLM 답변: {no_tool.content}")
print("→ LLM은 학습 데이터 기준 추측을 합니다. 정확하지 않습니다.")

# ─── B: current_date 도구 정의 ──────────────────────────────────────
@tool
def current_date() -> str:
    """
    현재 날짜를 조회합니다.

    사용 시점: 오늘 날짜·현재 시각이 필요할 때.
    예시 질문: '오늘 몇 월이야?', '지금 몇 년도야?', '이번 달이 뭐야?'

    Returns:
        str: YYYY-MM-DD 형식의 현재 날짜
    """
    return datetime.now().strftime("%Y-%m-%d")

print(f"\n=== B: current_date() 직접 호출 ===")
print(f"실제 오늘 날짜: {current_date.invoke({})}")

# ─── C: LLM + current_date 도구 조합 ──────────────────────────────────
print("\n=== C: current_date 도구 연결 후 같은 질문 ===")
llm_with_date = llm.bind_tools([current_date])
messages = [HumanMessage(content="오늘 날짜는?")]
ai_msg = llm_with_date.invoke(messages)

print(f"tool_calls: {ai_msg.tool_calls}")
messages.append(ai_msg)
for tc in ai_msg.tool_calls:
    result = current_date.invoke(tc["args"])
    messages.append(ToolMessage(content=str(result), tool_call_id=tc["id"]))

final = llm_with_date.invoke(messages)
print(f"최종 답변: {final.content}")
print("→ 이제 정확한 오늘 날짜를 답합니다.")

=== A: 도구 없이 질문 (LLM이 추측합니다) ===
LLM 답변: 오늘 날짜는 2023년 10월 4일입니다.
→ LLM은 학습 데이터 기준 추측을 합니다. 정확하지 않습니다.

=== B: current_date() 직접 호출 ===
실제 오늘 날짜: 2026-09-10

=== C: current_date 도구 연결 후 같은 질문 ===
tool_calls: [{'name': 'current_date', 'args': {}, 'id': 'call_PwR8GDYUpcuyaS3Lgasrsi2I', 'type': 'tool_call'}]
최종 답변: 오늘 날짜는 2026년 9월 10일입니다.
→ 이제 정확한 오늘 날짜를 답합니다.


## 🎯 연습 3 — simple_workflow 함수 완성하기

📖 강의 연계: 모듈 4-2 "v2: 도구 등록 & 단일 루프" / 모듈 4-4 "완전한 루프 구현"

4단계 루프를 반복 실행하는 `simple_workflow` 함수를 완성합니다.  
도구 호출이 더 이상 없을 때까지 `while` 루프로 반복하는 패턴입니다.

> 💡 이 패턴이 파이프라인 모듈에서 LangGraph ToolNode 한 줄로 자동화됩니다.

**TODO 3곳**: `tool_list` 딕셔너리 / `while` 조건 / `tool_exec` 꺼내기

In [18]:
# 🎯 연습 3 — simple_workflow 함수 TODO 채워 완성하기
# 📖 강의 연계: 모듈 4-2 "v2: 도구 등록 & 단일 루프 완성"
# TODO(🔰): 아래 ___ 3곳을 채우세요
#   ① tool_list: 도구 이름(str) → 도구 함수를 매핑하는 dict
#      힌트: {t.name: t for t in tools}
#   ② while 조건: tool_calls가 있는 동안 반복
#      힌트: ai_msg.tool_calls
#   ③ tool_exec: tool_list에서 도구 이름으로 함수를 꺼내기
#      힌트: tool_list[tool_name]

def simple_workflow(llm, question: str, tools: list) -> str:
    """
    질문과 도구 목록을 받아 도구 호출이 완료될 때까지 반복 실행합니다.
    """
    tool_list = {t.name: t for t in tools}           # TODO ① — 이름→함수 dict
    llm_wt = llm.bind_tools(tools)
    messages = [HumanMessage(content=question)]

    print(f"Q: {question}")
    ai_msg = llm_wt.invoke(messages)
    messages.append(ai_msg)

    while ai_msg.tool_calls:                  # TODO ② — ai_msg.tool_calls 조건
        for tc in ai_msg.tool_calls:
            tname = tc["name"]
            # print(f"  → 도구: {tname} | args: {tc['args']}")
            tool_exec = tool_list[tname]    # TODO ③ — tool_list에서 함수 꺼내기
            tool_result = tool_exec.invoke(tc)
            messages.append(tool_result)
        ai_msg = llm_wt.invoke(messages)
        messages.append(ai_msg)

    return ai_msg.content


# ── 테스트 ─────────────────────────────────────────────────────────────
tools_basic = [get_employee_info, calculate, current_date]

r1 = simple_workflow(llm, "EMP002 직원 정보 알려줘", tools_basic)
print(f"A: {r1}\n")

r2 = simple_workflow(llm, "1234 * 567은?", tools_basic)
print(f"A: {r2}\n")

r3 = simple_workflow(llm, "오늘 날짜는?", tools_basic)
print(f"A: {r3}")

Q: EMP002 직원 정보 알려줘
A: EMP002 직원 정보는 다음과 같습니다:

- 이름: 이영희
- 부서: 기획팀
- 직급: PM (프로젝트 매니저)

Q: 1234 * 567은?
A: 1234 * 567은 699678입니다.

Q: 오늘 날짜는?
A: 오늘 날짜는 2026년 9월 10일입니다.


## 🎯 연습 4 — 다단계 도구 실행: current_date → web_search 연속 호출

📖 강의 연계: 모듈 4-1 "4단계 흐름" / 모듈 4-3 "실제 외부 API 도구"

하나의 질문에 도구가 **순서대로 여러 번** 호출되는 다단계(Multi-step) 패턴을 관찰합니다.

"오늘 날짜랑 같은 월/일에 태어난 유명인은?" 질문은:
1. `current_date` → 오늘 날짜 확인
2. `web_search` → 그 날짜에 태어난 유명인 검색

두 도구가 연속으로 호출됩니다. LangSmith 트레이스에서 두 ToolMessage를 확인하세요.

In [19]:
# 🎯 연습 4 — 다단계 도구 실행 관찰
# 📖 강의 연계: 모듈 4-1 "4단계 흐름"

if TAVILY_AVAILABLE:
    tools_multi = [current_date, web_search]
    print("=== 다단계 도구 실행 ===")
    r = simple_workflow(llm, "오늘 날짜랑 같은 월/일에 태어난 유명인은?", tools_multi)
    print(f"\n최종 답변:\n{r[:400]}...")
    print("\n💡 위에서 current_date → web_search 순서로 두 번 호출됐나요?")
    print("   LangSmith 트레이스에서 ToolMessage 2개를 확인하세요.")
else:
    print("⚠️ 이 연습은 TAVILY_API_KEY가 필요합니다.")
    print("   대신 current_date 단독으로 다단계 없이 실행해봅니다:")
    print()
    r = simple_workflow(llm, "오늘이 몇 월 며칠인지 알려줘", [current_date])
    print(f"A: {r}")
    print()
    print("Tavily 키를 발급하면 current_date + web_search 연속 호출을 관찰할 수 있습니다.")

=== 다단계 도구 실행 ===
Q: 오늘 날짜랑 같은 월/일에 태어난 유명인은?

최종 답변:
9월 10일에 태어난 유명인들은 다음과 같습니다:

1. **Pope Julius III** (1487) - 교황
2. **Karl Lagerfeld** (1933) - 패션 디자이너
3. **Arnold Palmer** (1929) - 골프 선수
4. **Stephen Jay Gould** (1941) - 작가, 고생물학자
5. **Colin Firth** (1960) - 배우
6. **Joe Perry** (1950) - 음악가, Aerosmith의 기타리스트
7. **Ryan Phillippe** (1974) - 배우
8. **Guy Ritchie** (1968) - 감독

이 외에도 많은 유명인들이 9월 10일에 태어났습니다....

💡 위에서 current_date → web_search 순서로 두 번 호출됐나요?
   LangSmith 트레이스에서 ToolMessage 2개를 확인하세요.


---

## 🔰 기본 미션 — 내 마이 서비스 조각에 도구 연결하기

📖 강의 연계: 모듈 4-4 전체 (도구 설계 가이드 / 완전한 루프 구현)

**이것이 오늘의 핵심 산출물입니다.**  
Day 1부터 쌓아온 마이 서비스 조각에 `@tool` 도구를 연결해 완전한 루프를 완성합니다.  
이 결과물이 내일(9/11) 팀 빌딩 때 팀 주제의 재료가 됩니다.

---

### 서비스 유형별 도구 아이디어 (참고)

| 서비스 | 도구 후보 (Mock으로 구현 가능) |
|--------|------------------------------|
| 회의록 요약기 | `get_calendar_events`, `lookup_employee` |
| 이메일 도우미 | `get_contacts`, `search_email_history` |
| 민원 분류기 | `get_faq`, `lookup_policy` |
| 일정 조율 | `check_availability`, `find_meeting_room` |

> ℹ️ 실제 API 연결 없이 **Mock 데이터를 반환하는 함수**로 구현해도 됩니다.

---

In [ ]:
# 🔰 기본 미션 Step 1 — 내 서비스 도구 정의
# TODO(🔰): 아래 ___ 를 모두 채워 내 서비스 도구를 완성하세요
#   1. 함수 이름을 내 서비스에 맞게 변경 (예: lookup_faq, check_availability)
#   2. docstring의 '사용 시점:'과 Args를 구체적으로 작성
#   3. 간단한 Mock 데이터를 반환하도록 구현
#   (힌트: 📖 강의 모듈 4-4 '도구 설계 가이드 — v1' 예시 구조와 동일)

@tool
def ___(key: str) -> dict:  # ← 함수명을 내 서비스에 맞게 바꾸세요
    """
    ___ (한 줄 설명)

    사용 시점: ___

    Args:
        key: ___ (설명)
    Returns:
        dict: ___
    """
    # Mock 데이터 — 실제 서비스에서는 DB/API 연동
    data = {
        "___": {"field1": "___", "field2": "___"},
    }
    return data.get(key, {"error": f"데이터 없음: {key}"})

In [ ]:
# 🔰 기본 미션 Step 2 — run_service 완성 및 테스트
# TODO(🔰): 아래 ___ 를 채워 완전한 루프를 완성하세요
#   1. bind_tools에 위에서 정의한 내 도구 함수를 넣으세요
#   2. SystemMessage content를 내 서비스에 맞는 시스템 프롬프트로 작성하세요
#   3. test_questions 3개를 내 서비스 시나리오에 맞게 작성하세요
#   (힌트: 📖 강의 모듈 4-4 '완전한 루프 구현 — v2: 서비스 함수' 참조)

my_llm = llm.bind_tools([___])  # ← 위에서 만든 내 도구 함수 이름

def run_my_service(user_question: str) -> str:
    """내 서비스 메인 실행 함수"""
    messages = [
        SystemMessage(content="___"),  # ← 내 서비스 시스템 프롬프트 작성
        HumanMessage(content=user_question),
    ]
    ai_msg = my_llm.invoke(messages)
    messages.append(ai_msg)

    if ai_msg.tool_calls:
        for tc in ai_msg.tool_calls:
            result = ___.invoke(tc["args"])  # ← 내 도구 함수 이름
            messages.append(ToolMessage(
                content=str(result), tool_call_id=tc["id"],
            ))
        final = my_llm.invoke(messages)
        return final.content
    else:
        return ai_msg.content

# 테스트 — 도구 사용 경로 2개 + 직접 답변 1개
test_questions = [
    "___",   # 도구를 사용해야 하는 질문 1
    "___",   # 도구를 사용해야 하는 질문 2
    "___",   # 도구 없이 직접 답변하는 질문
]

print("=== 내 서비스 테스트 ===")
for q in test_questions:
    print(f"Q: {q}")
    print(f"A: {run_my_service(q)}")
    print()

---

## ⭐ 심화 미션 — 예외 안전 처리 + 연속 대화 (기본 미션 완료 후 진행)

📖 강의 연계: 모듈 4-4 '⭐ 심화: 병렬 tool_calls & 예외 처리'

**요구사항 ①: 예외 안전 처리**  
도구 실행이 실패해도 서비스가 중단되지 않도록 `try-except`를 추가하고  
오류를 `ToolMessage`로 AI에게 전달해 대체 답변을 유도하세요.

**요구사항 ②: 연속 대화 이력 유지**  
`run_my_service`에 `chat_history` 파라미터를 추가해  
앞 질문의 맥락을 기억하는 연속 대화가 되도록 개선하세요.  
(이력은 메모리 리스트로만 관리 — 파이프라인 모듈의 DB 영속화 예고편)

완료 시 슬랙 `#day4-제출` 채널에 심화 결과 LangSmith 링크를 올려주세요.

In [ ]:
# ⭐ 심화 미션 — 아래에 직접 구현하세요 (힌트 없음)
# 기본 미션의 run_my_service를 발전시킵니다.

---

## ⭐ 심화 미션 ③ — 논문 검색 도구 추가 (ArXiv)

📖 강의 연계: 모듈 4-3 "⭐ 실습 4번: 논문 검색 도구 추가"

`langchain_community`의 `ArxivQueryRun`을 `@tool`로 래핑해
arXiv 학술 논문 검색 도구를 `simple_workflow`에 추가합니다.

**요구사항**
- `paper_search` @tool 구현 — docstring에 "사용 시점:" 명확히 작성
- `web_search`, `news_search`, `paper_search` 3개 도구를 동시에 등록
- "LangGraph 논문 찾아줘"로 테스트 → `paper_search`가 선택되는지 확인
- docstring을 조정해 선택 정확도를 높여보세요

완료 시 슬랙 `#day4-제출`에 심화 결과 LangSmith 링크를 올려주세요.

In [ ]:
# ⭐ 심화 미션 ③ — 논문 검색 도구 (ArXiv) 직접 구현
# 📖 강의 연계: 모듈 4-3 "⭐ 심화 실습 4번: 논문 검색 도구 추가 (ArXiv)"
# pip install langchain_community 필요

# TODO(⭐): 아래 paper_search를 완성하고 3도구 조합으로 테스트하세요

try:
    from langchain_community.tools import ArxivQueryRun

    @tool
    def paper_search(query: str) -> str:
        """
        arXiv에서 AI·ML·CS 학술 논문을 검색합니다.

        사용 시점: ___
        사용하지 말 것: ___

        Args:
            query: 논문 검색어 (영어 권장, 예: 'ReAct prompting', 'LangGraph multi-agent')
        """
        return ArxivQueryRun().run(query)

    # 3도구 조합 등록
    llm_research = llm.bind_tools([web_search, news_search, paper_search])

    test_research = [
        "LangGraph 논문 찾아줘",           # 예상: paper_search
        "GPT-5 출시 관련 최신 뉴스",        # 예상: news_search
        "LangChain 최신 버전 기능",         # 예상: web_search
    ]

    print("3도구(web / news / paper) 선택 실험:")
    print("=" * 55)
    for q in test_research:
        msg = llm_research.invoke([HumanMessage(content=q)])
        chosen = msg.tool_calls[0]["name"] if msg.tool_calls else "직접 답변"
        print(f"Q: {q}")
        print(f"   → {chosen}\n")

except ImportError:
    print("⚠️  langchain_community 미설치")
    print("   pip install langchain_community 실행 후 재시도하세요")

---

## ⭐ 심화 미션 ② — 내 경험 기반 자유 주제 구현

📖 강의 연계: 모듈 4-4 전체 + Day 1~3 마이 서비스 조각

이번 심화는 **정해진 주제가 없습니다.**  
여러분의 업무·학교·일상에서 AI + 도구 연결로 해결하고 싶은 문제를 직접 골라 구현합니다.  
오늘까지 배운 4단계 루프, docstring 설계, 분기 처리를 모두 녹여내는 자유 도전입니다.

---

### 💡 아이디어 발굴 질문 — 먼저 스스로에게 물어보세요

> 1. **반복 작업**: 매주 같은 형식으로 처리하는 귀찮은 일이 있나요?  
>    (예: 회의록 정리, 일간 보고 초안, 민원 이메일 분류)
>
> 2. **정보 검색**: 자주 찾아보는 정보인데 매번 검색이 번거로운 것이 있나요?  
>    (예: 사내 규정, 제품 스펙, 업무 연락처, 내부 FAQ)
>
> 3. **판단·분류**: '이게 자동화되면 좋겠다' 싶었던 결정 작업이 있나요?  
>    (예: 우선순위 정하기, 문의 유형 분류, 리뷰 감성 분석)

---

### 📐 설계 체크리스트 — 구현 전 **설계 메모 셀**을 먼저 채우세요

| 항목 | 내용 |
|------|------|
| 서비스 이름 | ___ |
| 해결하려는 문제 | ___ |
| 도구가 조회할 정보 | ___ (예: 사내 규정, 재고, 일정) |
| 도구 입력 파라미터 | ___ (이름 + 타입, 예: `keyword: str`) |
| 도구 반환 구조 | ___ (예: `{"result": str, "link": str}`) |
| 테스트 질문 3개 | ① ___ ② ___ ③ ___ |

> ⚠️ **docstring 품질 자가 점검** (구현 후 이 질문에 답할 수 있어야 합니다)
> - 사용 시점을 1문장으로 쓸 수 있는가?
> - 예시 질문을 2개 이상 쓸 수 있는가?
> - 파라미터 타입·설명이 명확한가?
> - description이 없으면 AI가 이 도구를 선택할 수 있겠는가?

In [ ]:
# ⭐ 심화 미션 ② — Step 0: 설계 메모 (구현 전 반드시 채우세요)
# 이 셀은 실행하지 않습니다. 주석을 채워 아이디어를 구체화하세요.

# ── 서비스 개요 ──────────────────────────────────────────────────────
# 서비스 이름       : ___
# 해결하는 문제     : ___
# 대상 사용자       : ___  (예: 우리 팀 / 나 자신 / 고객)

# ── 도구 설계 ────────────────────────────────────────────────────────
# 도구 이름(함수명) : ___  (예: search_inventory, lookup_schedule)
# 도구가 하는 일    : ___  (한 문장)
# 사용 시점         : ___  (AI가 언제 이 도구를 골라야 하나?)
# 예시 질문 2개     : (1) ___ / (2) ___
# 파라미터 이름·타입: ___: ___  (예: keyword: str)
# 반환값 구조       : {"___": "___", "___": "___"}

# ── 테스트 시나리오 ──────────────────────────────────────────────────
# 도구 사용 질문 ①  : ___
# 도구 사용 질문 ②  : ___
# 직접 답변 질문    : ___  (도구 없이 AI가 직접 답하는 케이스)

# ── 시스템 프롬프트 초안 ──────────────────────────────────────────────
# "당신은 ___ 도우미입니다. ___에 대한 질문에 ___ 어조로 답해주세요."

In [ ]:
# ⭐ 심화 미션 ② — Step 1~4: 자유 주제 구현
# 위 설계 메모를 보며 아래 4개 섹션을 채우세요.
# 📖 강의 연계: 모듈 4-4 '도구 설계 가이드 v1 + v2' 구조와 동일

# ─── 1. 도구 정의 (@tool) ────────────────────────────────────────────
@tool
def my_custom_tool(param: str) -> dict:
    """
    (한 줄 설명)

    사용 시점: (언제 이 도구를 써야 하나? 1문장)
    예시 질문: (예시 질문 2개 이상)

    Args:
        param: (타입과 설명, 설계 메모의 파라미터 이름으로 변경)
    Returns:
        dict: (반환 구조 설명)
    """
    # Mock 데이터 — 실제 서비스에서는 실제 DB/API 연동
    data = {
        # "키": {"필드1": "값", "필드2": "값"},  ← 여기에 예시 데이터를 추가하세요
    }
    return data.get(param, {"error": f"데이터 없음: {param}"})


# ─── 2. LLM + 도구 등록 ──────────────────────────────────────────────
free_llm = llm.bind_tools([my_custom_tool])


# ─── 3. 서비스 함수 ───────────────────────────────────────────────────
def run_free_service(user_question: str) -> str:
    """자유 주제 서비스 실행 함수"""
    messages = [
        SystemMessage(content="당신은 ___ 도우미입니다."),  # ← 시스템 프롬프트 작성
        HumanMessage(content=user_question),
    ]
    ai_msg = free_llm.invoke(messages)
    messages.append(ai_msg)

    if ai_msg.tool_calls:
        for tc in ai_msg.tool_calls:
            result = my_custom_tool.invoke(tc["args"])
            messages.append(ToolMessage(
                content=str(result), tool_call_id=tc["id"]
            ))
        final = free_llm.invoke(messages)
        return final.content
    else:
        return ai_msg.content


# ─── 4. 테스트 실행 ───────────────────────────────────────────────────
my_questions = [
    "___",  # ← 도구 사용 경로: 설계 메모의 테스트 질문 ① 입력
    "___",  # ← 도구 사용 경로: 설계 메모의 테스트 질문 ② 입력
    "___",  # ← 직접 답변 경로: 도구 불필요 질문 입력
]

print("=== 내 자유 주제 서비스 테스트 ===")
for q in my_questions:
    print(f"Q: {q}")
    print(f"A: {run_free_service(q)}")
    print()

# ─── 5. 성찰 메모 ────────────────────────────────────────────────────
# 이 도구가 실제 내 업무·생활에 있다면 어떤 점이 달라질까? (1~2문장)
# ___

---

## 📬 제출 & 자가 체크

### ✅ 완료 확인 목록

- [ ] **Step 1**: `args_schema`를 출력해 docstring → 스키마 변환을 확인했다
- [ ] **Step 2**: `tool_calls` JSON 구조(name / args / id)를 셀 출력으로 직접 읽었다
- [ ] **Step 3**: 도구 사용 경로와 직접 답변 경로 양쪽을 테스트했다
- [ ] **Step 4**: description 짧을 때 vs 충분할 때 선택 결과 차이를 확인했다
- [ ] **연습 1**: `kwargs['tools']`로 AI에게 전달되는 JSON 스키마를 확인했다
- [ ] **연습 2**: LLM이 오늘 날짜를 모른다는 것을 확인하고 `current_date` 도구로 해결했다
- [ ] **연습 3**: `simple_workflow` TODO 3곳을 채워 정상 실행했다
- [ ] **🔰 기본 미션**: 내 서비스 도구 1개 이상 + `run_my_service()` 완전한 루프가 동작한다
- [ ] **(⭐ 선택) Step 5**: Tavily 웹 검색·뉴스 도구가 4도구 조합에서 올바르게 선택된다
- [ ] **제출**: LangSmith 트레이스 링크를 슬랙 `#day4-제출` 채널에 올렸다

### 📤 제출

슬랙 `#day4-제출` 채널에 아래 내용을 올려주세요:

```
[Day 4 실습 제출]
마이 서비스 이름: ___
내가 만든 도구: ___ (함수명)
테스트 질문 예시: ___
LangSmith 링크: ___
(선택) Tavily 도구 구현 여부: Y/N
(선택) 심화 완료 여부: Y/N
```

---
> ➡️ **Day 5 예고**: 내일은 팀 빌딩 데이입니다. 오늘 만든 마이 서비스 조각을 들고 오세요!